# 🧬 Neuro-CRISPR-KAN: Full Pipeline Runner
**Hybrid CNN-Transformer Architecture for Off-Target Prediction in Cystic Fibrosis**

IEEE ICAUC 2026 | Paper ID: ICAUC-500

---

This notebook runs the complete pipeline in order:
1. Setup & Installation
2. Data Generation (10K synthetic samples)
3. Encoding & DataLoaders
4. Model Building
5. Training
6. Evaluation & Ablation
7. RAG Safety Audit Generation
8. Visualization

## 1. Setup & Installation

In [ ]:
# Install dependencies
!pip install -q torch transformers peft sentence-transformers chromadb \
    scikit-learn scipy matplotlib seaborn plotly streamlit tqdm pandas numpy

In [ ]:
# If running from Colab, upload the project folder or clone from git
# !git clone https://github.com/YOUR_USERNAME/neuro-crispr-kan.git
# %cd neuro-crispr-kan

import sys
import os

# Add project root to path
PROJECT_ROOT = os.getcwd()  # Adjust if needed
sys.path.insert(0, PROJECT_ROOT)

print(f'Project root: {PROJECT_ROOT}')
print(f'Files: {os.listdir(PROJECT_ROOT)}')

In [ ]:
# Core imports
import torch
import numpy as np
import pandas as pd

from configs.config import config
from utils.helpers import set_seed, get_device, count_parameters

# Set seed for reproducibility
set_seed(config.seed)
device = get_device()
print(f'Config loaded. Device: {device}')

## 2. Data Generation

In [ ]:
from data.data_generation import generate_dataset, save_dataset

# Generate 10,000 synthetic sgRNA-DNA pairs
df = generate_dataset()
save_dataset(df)

# Preview
df.head(10)

In [ ]:
# Dataset distribution
print(f"Label distribution:\n{df['off_target_label'].value_counts()}")
print(f"\nDeletion distribution:\n{df['has_deletion'].value_counts()}")
print(f"\nMismatch stats:\n{df['num_mismatches'].describe()}")

## 3. Encoding & DataLoaders

In [ ]:
from data.encoding import create_dataloaders, NullTensorEncoder

# Create DataLoaders with Null Tensor encoding
loaders = create_dataloaders(
    df, 
    encoder='null_tensor', 
    batch_size=config.training.batch_size
)

# Verify shapes
batch = next(iter(loaders['train']))
print(f"Encoded shape: {batch['encoded'].shape}")  # (batch, 2, 23, 5)
print(f"Labels shape:  {batch['label'].shape}")     # (batch, 1)
print(f"sgRNA example: {batch['sgrna_seq'][0][:20]}...")

## 4. Build Model

In [ ]:
from models.neuro_crispr_kan import NeuroCRISPRKAN

# Build the full model
print('Building Neuro-CRISPR-KAN...')
model = NeuroCRISPRKAN()
model = model.to(device)

# Parameter summary
params = count_parameters(model)
print(f"\nTotal parameters: {params['total']:,}")
print(f"Trainable: {params['trainable']:,} ({params['trainable_pct']:.1f}%)")
print(f"Frozen: {params['frozen']:,}")

In [ ]:
# Test forward pass
with torch.no_grad():
    test_batch = next(iter(loaders['val']))
    outputs = model(
        test_batch['encoded'].to(device),
        test_batch['sgrna_seq'],
        test_batch['dna_seq'],
        device=device
    )
    print('Forward pass successful!')
    for k, v in outputs.items():
        if isinstance(v, torch.Tensor):
            print(f'  {k}: {v.shape}')

## 5. Training

In [ ]:
from training.train import train

# Train the model
history = train(
    model=model,
    train_loader=loaders['train'],
    val_loader=loaders['val'],
    device=device
)

print(f'\nTraining complete! {len(history)} epochs.')

## 6. Evaluation

In [ ]:
from evaluation.evaluate import evaluate_model
from evaluation.visualize import (
    plot_training_curves, plot_confusion_matrix, 
    plot_roc_curve, plot_comparison_bars
)

# Load best model
from utils.helpers import load_checkpoint
load_checkpoint(model, None, './checkpoints/best_model.pt', device)

# Evaluate on test set
results = evaluate_model(model, loaders['test'], device)

# Plot training curves
plot_training_curves(history)

In [ ]:
# Confusion matrix
plot_confusion_matrix(
    results['raw']['labels'], 
    results['raw']['predictions']
)

# ROC curve
plot_roc_curve(
    results['raw']['labels'], 
    results['raw']['predictions']
)

In [ ]:
# Comparison chart (using paper's baseline numbers)
comparison = {
    'DeepCRISPR': {'accuracy': 0.87, 'precision': 0.84, 'recall': 0.81, 'f1_score': 0.82},
    'CRISPR-Net': {'accuracy': 0.91, 'precision': 0.89, 'recall': 0.85, 'f1_score': 0.87},
    'Neuro-CRISPR-KAN': results['classification'],
}
plot_comparison_bars(comparison)

## 7. RAG Safety Audit

In [ ]:
from rag.rag_llm import RAGModule, generate_template_audit

# Quick template-based audit (no LLM needed)
audit = generate_template_audit(
    risk_score=0.85,
    num_mismatches=1,
    seed_mismatches=0,
    has_deletion=True,
    chromatin_score=0.7,
    pam_intact=True,
)
print(audit)

In [ ]:
# Full RAG-LLM audit (downloads models ~1GB)
# Uncomment to run:

# rag = RAGModule()
# rag.initialize()
# report = rag.generate_safety_audit(
#     sgrna_seq='ATCGATCGATCGATCGATCGNGG',
#     dna_seq='ATCGATCGATCGATCGATCGNGG',
#     risk_score=0.85,
#     num_mismatches=1,
#     seed_mismatches=0,
#     has_deletion=True,
#     chromatin_score=0.7,
#     pam_intact=True,
# )
# print(report)

## 8. Summary

### Results achieved:
- ✅ Synthetic dataset generated (10K samples)
- ✅ Null Tensor encoding implemented
- ✅ Hybrid CNN + DNABERT-2 (LoRA) feature extraction
- ✅ KAN decision core with B-spline activations
- ✅ Focal loss + spline regularization training
- ✅ Full evaluation with paper metrics
- ✅ RAG-based safety audit generation
- ✅ Streamlit UI ready (`streamlit run ui/app.py`)